In [ ]:
from google.colab import drive
import os

# 1. Mount Google Drive
drive.mount('/content/drive', force_remount=True)

# 2. Define your paths
# Note: Ensure 'ravdess.zip' is in your 'MyDrive' folder
zip_source = '/content/drive/MyDrive/ravdess.zip'
destination = '/content/data/raw/'

# 3. Create the destination directory
print(f"Creating directory: {destination}")
!mkdir -p {destination}

# 4. Check for the Zip file and Unzip (The Fast Method)
if os.path.exists(zip_source):
    print("Found 'ravdess.zip'. Unzipping now... (This is fast)")

    # -q means 'quiet' (hides the long list of file names)
    # -d specifies the destination folder
    !unzip -q {zip_source} -d {destination}

    print("Success! RAVDESS dataset unzipped.")

else:
    # Fallback or Error message
    print(f"⚠️ Could not find: {zip_source}")
    print("Please make sure you have zipped your 'ravdess' folder and uploaded it to Drive.")
    print("If you really want to copy the unzipped folder (slow), uncomment the line below:")
    # !cp -r /content/drive/MyDrive/ravdess /content/data/raw/

# 5. Verify the data
print("\nVerifying data...")
!ls -lh {destination}

# Count how many audio files are inside
print("Counting audio files (this may take a moment)...")
!find {destination} -type f \\( -name '*.wav' -o -name '*.mp3' \\) | wc -l

## 1. GPU Setup & Environment

In [ ]:
from google.colab import drive
import os

# 1. Mount Google Drive
drive.mount('/content/drive', force_remount=True)

# 2. Define your paths
zip_source = '/content/drive/MyDrive/ravdess.zip'
destination = '/content/data/raw/'

# 3. Create the destination directory
print(f"Creating directory: {destination}")
!mkdir -p {destination}

# 4. Check for the Zip file and Unzip
if os.path.exists(zip_source):
    print("Found 'ravdess.zip'. Unzipping now... (This is fast)")
    !unzip -q {zip_source} -d {destination}
    print("Success! RAVDESS dataset unzipped.")
else:
    print(f"⚠️ Could not find: {zip_source}")
    print("Please upload ravdess.zip to your Google Drive MyDrive folder.")

# 5. Verify the data
print("\nVerifying data...")
!ls -lh {destination}
print("\nCounting audio files...")
!find {destination} -type f \( -name '*.wav' -o -name '*.mp3' \) | wc -l

In [ ]:
# Check GPU
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    print("✓ GPU ready for training")

In [ ]:
# Install required packages
!pip install -q librosa soundfile transformers torch torchaudio scikit-learn gradio tqdm 2>&1 | tail -5
print("✓ Packages installed")

In [ ]:
# Imports
import os
import numpy as np
import librosa
import soundfile as sf
from pathlib import Path
import matplotlib.pyplot as plt
from tqdm import tqdm
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from transformers import AutoFeatureExtractor, AutoModelForAudioClassification
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import gradio as gr
from datetime import datetime

print("✓ All imports successful")

## 2. RAVDESS Dataset Loading

In [ ]:
# Load RAVDESS dataset - CORRECTED
def load_ravdess(root_dir='/content/data/raw/ravdess'):
    """Load RAVDESS audio files and extract emotion labels"""
    # RAVDESS emotions: 01=neutral, 02=calm, 03=happy, 04=sad, 05=angry, 06=fearful, 07=disgust, 08=surprised
    emotion_map = {
        '01': 'neutral',
        '02': 'calm',
        '03': 'happy',
        '04': 'sad',
        '05': 'angry',
        '06': 'fearful',
        '07': 'disgust',
        '08': 'surprised'
    }

    samples = []

    # Find all .wav files in the ravdess directory
    audio_files = list(Path(root_dir).glob('**/*.wav'))

    print(f"Found {len(audio_files)} RAVDESS audio files")

    for audio_file in audio_files:
        # Extract emotion from filename
        # Format: 03-01-06-01-43-01-12.wav
        parts = audio_file.stem.split('-')
        if len(parts) >= 3:
            emotion_code = parts[2]
            if emotion_code in emotion_map:
                emotion = emotion_map[emotion_code]
                samples.append({
                    'path': str(audio_file),
                    'emotion': emotion,
                    'emotion_code': emotion_code
                })

    # Count emotions
    emotion_counts = {}
    for sample in samples:
        emotion = sample['emotion']
        emotion_counts[emotion] = emotion_counts.get(emotion, 0) + 1

    print("\nEmotion distribution:")
    for emotion, count in sorted(emotion_counts.items()):
        print(f"  {emotion}: {count} files")

    return samples, emotion_map

# Load dataset
ravdess_samples, emotion_map = load_ravdess()
print(f"\n✓ Total samples: {len(ravdess_samples)}")

## 3. Audio Dataset Class

In [ ]:
class RAVDESSDataset(Dataset):
    """RAVDESS Audio Dataset"""
    def __init__(self, samples, feature_extractor, sample_rate=16000):
        self.samples = samples
        self.feature_extractor = feature_extractor
        self.sample_rate = sample_rate

        # Create emotion to index mapping
        emotions = sorted(set(s['emotion'] for s in samples))
        self.emotion2idx = {e: i for i, e in enumerate(emotions)}
        self.idx2emotion = {i: e for e, i in self.emotion2idx.items()}
        self.emotions = emotions

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]

        # Load audio
        try:
            audio, sr = librosa.load(sample['path'], sr=self.sample_rate)
        except Exception as e:
            print(f"Error loading {sample['path']}: {e}")
            audio = np.zeros(self.sample_rate * 3)  # 3 second silence

        # Feature extraction
        inputs = self.feature_extractor(audio, sampling_rate=self.sample_rate, return_tensors="pt", padding=True)

        label = self.emotion2idx[sample['emotion']]

        return {
            'input_values': inputs['input_values'].squeeze(0),
            'attention_mask': inputs.get('attention_mask', torch.ones(inputs['input_values'].shape[-1])).squeeze(0) if 'attention_mask' in inputs else torch.ones(inputs['input_values'].shape[-1]),
            'label': torch.tensor(label, dtype=torch.long)
        }

print("✓ Dataset class defined")

## 4. Load Model and Create DataLoaders

In [ ]:
# Load HuBERT feature extractor and model (Publicly available - no auth required)
feature_extractor = AutoFeatureExtractor.from_pretrained('facebook/hubert-large-ls960-ft')

# Custom collate function to handle variable-length sequences
def collate_fn(batch):
    """Pad sequences to same length and stack them"""
    max_length = max(item['input_values'].shape[0] for item in batch)
    
    padded_input_values = []
    attention_masks = []
    labels = []
    
    for item in batch:
        # Get input values
        input_vals = item['input_values']
        current_length = input_vals.shape[0]
        
        # Pad to max length
        if current_length < max_length:
            padding = torch.zeros(max_length - current_length)
            padded_input = torch.cat([input_vals, padding])
        else:
            padded_input = input_vals
        
        # Create attention mask (1 for real values, 0 for padding)
        attention_mask = torch.ones(max_length)
        if current_length < max_length:
            attention_mask[current_length:] = 0
        
        padded_input_values.append(padded_input)
        attention_masks.append(attention_mask)
        labels.append(item['label'])
    
    return {
        'input_values': torch.stack(padded_input_values),
        'attention_mask': torch.stack(attention_masks),
        'label': torch.stack(labels)
    }

# Create datasets
# Split data: 80% train, 20% test
from sklearn.model_selection import train_test_split

train_samples, test_samples = train_test_split(ravdess_samples, test_size=0.2, random_state=42, stratify=[s['emotion'] for s in ravdess_samples])

train_dataset = RAVDESSDataset(train_samples, feature_extractor)
test_dataset = RAVDESSDataset(test_samples, feature_extractor)

emotions = train_dataset.emotions
num_emotions = len(emotions)

# Create dataloaders with custom collate function
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, num_workers=0, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False, num_workers=0, collate_fn=collate_fn)

print(f"✓ Train samples: {len(train_dataset)}")
print(f"✓ Test samples: {len(test_dataset)}")
print(f"✓ Emotions: {emotions}")
print(f"✓ Number of emotion classes: {num_emotions}")

In [ ]:
# Load HuBERT model for audio classification
model = AutoModelForAudioClassification.from_pretrained(
    'facebook/hubert-large-ls960-ft',
    num_labels=num_emotions,
    ignore_mismatched_sizes=True
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

print(f"✓ HuBERT model loaded on {device}")

## 5. Training Functions

In [ ]:
def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    for batch in tqdm(loader, desc="Training"):
        input_values = batch['input_values'].to(device)
        labels = batch['label'].to(device)

        optimizer.zero_grad()
        outputs = model(input_values, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    return total_loss / len(loader)

def eval_epoch(model, loader, device):
    model.eval()
    all_preds, all_labels = [], []
    total_loss = 0

    with torch.no_grad():
        for batch in tqdm(loader, desc="Evaluating"):
            input_values = batch['input_values'].to(device)
            labels = batch['label'].to(device)

            outputs = model(input_values, labels=labels)
            loss = outputs.loss
            total_loss += loss.item()

            preds = torch.argmax(outputs.logits, dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())

    acc = accuracy_score(all_labels, all_preds)
    return acc, np.array(all_preds), np.array(all_labels), total_loss / len(loader)

print("✓ Training functions defined")

## 6. Train HuBERT

In [ ]:
# Training parameters
num_epochs = 10
learning_rate = 1e-4

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=learning_rate)

print("\n=== HuBERT Speech Emotion Training ===")
for epoch in range(num_epochs):
    train_loss = train_epoch(model, train_loader, optimizer, criterion, device)
    test_acc, _, _, test_loss = eval_epoch(model, test_loader, device)
    print(f"Epoch {epoch+1}/{num_epochs} | Train Loss: {train_loss:.4f} | Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.4f}")

print("\n✓ HuBERT training complete")

## 7. Model Evaluation

In [ ]:
# Final evaluation
model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for batch in test_loader:
        input_values = batch['input_values'].to(device)
        labels = batch['label'].to(device)

        outputs = model(input_values)
        preds = torch.argmax(outputs.logits, dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

# Metrics
accuracy = accuracy_score(all_labels, all_preds)
precision = precision_score(all_labels, all_preds, average='weighted', zero_division=0)
recall = recall_score(all_labels, all_preds, average='weighted', zero_division=0)
f1 = f1_score(all_labels, all_preds, average='weighted', zero_division=0)

print("\n=== HuBERT Speech Emotion Performance ===")
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-Score:  {f1:.4f}")

In [ ]:
# Confusion matrix
cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(10, 8))
plt.imshow(cm, cmap='Blues', interpolation='nearest')
plt.title('Confusion Matrix - HuBERT Speech Emotion Recognition')
plt.colorbar()
plt.xticks(range(len(emotions)), emotions, rotation=45)
plt.yticks(range(len(emotions)), emotions)
plt.xlabel('Predicted')
plt.ylabel('Actual')

# Add text annotations
for i in range(len(emotions)):
    for j in range(len(emotions)):
        plt.text(j, i, str(cm[i, j]), ha='center', va='center', color='white' if cm[i, j] > cm.max() / 2 else 'black')

plt.tight_layout()
plt.show()

print("✓ Confusion matrix displayed")

## 8. Save Model

In [ ]:
# Save model
model_dir = Path('/content/models/phase3')
model_dir.mkdir(parents=True, exist_ok=True)

model_path = model_dir / 'hubert_emotion_model.pt'
torch.save({
    'model_state_dict': model.state_dict(),
    'accuracy': accuracy,
    'timestamp': datetime.now().isoformat()
}, model_path)

print(f"✓ Model saved to {model_path}")

## 9. Interactive Gradio Demo

In [ ]:
def predict_speech_emotion(audio):
    """Predict emotion from audio"""
    if audio is None:
        return {emotion: 0 for emotion in emotions}

    try:
        # Load audio
        sr = audio[0]
        audio_data = audio[1]

        # Resample if needed
        if sr != 16000:
            audio_data = librosa.resample(audio_data, orig_sr=sr, target_sr=16000)

        # Feature extraction
        inputs = feature_extractor(audio_data, sampling_rate=16000, return_tensors="pt", padding=True)

        # Predict
        model.eval()
        with torch.no_grad():
            outputs = model(inputs['input_values'].to(device))
            logits = outputs.logits.cpu().numpy()[0]
            probs = np.exp(logits) / np.sum(np.exp(logits))

        # Return probabilities
        return {emotion: float(prob) for emotion, prob in zip(emotions, probs)}

    except Exception as e:
        print(f"Error: {e}")
        return {emotion: 0 for emotion in emotions}

# Create interface
demo = gr.Interface(
    fn=predict_speech_emotion,
    inputs=gr.Audio(label='Upload or Record Audio'),
    outputs=gr.Label(label='Emotion Probabilities', num_top_classes=len(emotions)),
    title='Speech Emotion Recognition (Phase 3)',
    description='Upload or record audio to predict emotion using HuBERT'
)

print("✓ Gradio demo ready")

In [ ]:
# Launch demo
demo.launch(share=True)

In [ ]:
# Download model
from google.colab import files
files.download('/content/models/phase3/hubert_emotion_model.pt')